# Benchmark Radar Detectors

Compares fixed-threshold, CA-CFAR, and OS-CFAR operating points using raw-data injections and the complete 24-hour campaign. It freezes the detector configuration used downstream.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


### Swerling cases implemented


### Fast checkpoint resume


In [ ]:
from pathlib import Path

HARDTARGET_COMMIT = "ffdf6cee99f021d554c34dec50360cf614864896"
import numpy as np
import scipy

RESUME_REAL_ONLY = True
print("FAST RESUME MODE: no package installation and no Hardtarget compile.")
print("NumPy:", np.__version__, "SciPy:", scipy.__version__)
print("Completed raw campaign commit retained:", HARDTARGET_COMMIT)


In [ ]:
print("Repository paths configured.", flush=True)


In [ ]:
from __future__ import annotations
import gc
import hashlib
import json
import math
import multiprocessing as mp
import os
import re
import shutil
import subprocess
import time
import traceback
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from scipy.ndimage import maximum_filter1d
from scipy.signal import savgol_filter
from scipy.stats import chi2, expon, gamma as gamma_distribution, kstest
BASE = PROJECT_ROOT
PRODUCTION_ROOT = BASE / 'processed/tromso_2018/gmf'
LEGACY_HOUR_DIR = BASE / 'processed/tromso_2018/legacy_development/20180104_12/gmf_full_hour'
OUTPUT_ROOT = BASE / 'phase0_outputs/detector_benchmark_raw_swerling_v2'
HOURLY_ROOT = OUTPUT_ROOT / 'hourly'
SWERLING_ROOT = OUTPUT_ROOT / 'swerling_raw'
LOCAL_STAGE_ROOT = Path('.cache/debris_detector_benchmark_stage')
LOCAL_SWERLING_ROOT = Path('.cache/debris_swerling_raw_batches')
COHINT_SEC = 0.2
N_IPP = 10
IPP_SAMPLES = 20000
SAMPLE_RATE_HZ = 1000000
IPP_SEC = IPP_SAMPLES / SAMPLE_RATE_HZ
assert abs(N_IPP * IPP_SEC - COHINT_SEC) < 1e-12
SNR_SMOOTH_WINDOW = 101
TRAIN_CELLS = 32
GUARD_CELLS = 16
OS_ORDER_FRACTION = 0.75
EPS = np.finfo(np.float32).tiny
FIXED_THRESHOLDS_DB = [10.0, 11.0, 12.0, 13.0, 13.9794000867, 15.0, 17.5, 20.0, 22.5, 25.0]
CFAR_THRESHOLDS = [1.5, 1.75, 2.0, 2.5, 3.0, 4.0, 5.0]
TRACK_RULES = [{'track_rule': 'short_0p4', 'min_unique_times': 3, 'min_duration_sec': 0.4}, {'track_rule': 'short_0p6', 'min_unique_times': 4, 'min_duration_sec': 0.6}, {'track_rule': 'short_0p8', 'min_unique_times': 5, 'min_duration_sec': 0.8}, {'track_rule': 'legacy_1p0', 'min_unique_times': 6, 'min_duration_sec': 1.0}]
MAX_LINK_GAP_STEPS = 4
MAX_ABS_RADIAL_SPEED_KM_S = 6.0
LINK_RANGE_PADDING_KM = 0.45
MAX_TRACK_RMSE_KM = 0.45
NULL_TRIALS_PER_HOUR = 25
NULL_BLOCK_STEPS = 2
NULL_TRIALS_TARGET_PER_HOUR = 4
NULL_WORKERS = 4
SWERLING_MODELS = ['Swerling0', 'Swerling1', 'Swerling2', 'Swerling3', 'Swerling4']
SWERLING_OUTPUT_SNR_DB = [10.0, 12.0, 14.0, 16.0, 18.0]
SWERLING_DURATIONS_SEC = [0.4, 0.6, 0.8, 1.0, 1.5, 2.0, 4.0, 8.0]
SWERLING_REPLICATES_PER_CELL = 20
SWERLING_BATCH_SIZE = 40
SWERLING_QUIET_GAP_SEC = 1.2
SWERLING_BATCH_EDGE_PAD_SEC = 1.0
SWERLING_PHASE_MODEL = 'range_coherent_rcs_only'
SWERLING_PILOT_INPUT_SNR_DB = -10.0
SWERLING_SPEED_CHOICES_KM_S = [-2.0, -1.0, -0.5, 0.5, 1.0, 2.0]
SWERLING_ACCELERATION_CHOICES_KM_S2 = [-0.05, 0.0, 0.05]
SWERLING_EXPERIMENT = {'sample_rate': SAMPLE_RATE_HZ, 'ipp': 20000, 'tx_pulse_length': 1920.0, 'tx_start': 82.0, 'tx_end': 2002.0, 'rx_start': 0, 'rx_end': 20000, 'cal_on': 19900.0, 'cal_off': 19997.0, 'frequency': 929.6, 'baud_length': 30.0, 'code_name': 'leo_bpark'}
SWERLING_GMF_CONFIG_TEXT = '[signal-processing]\nn_ipp = 10\nipp_offset = 0\nmin_range_gate = 6800\nmax_range_gate = 7280\nrange_gate_step = 1\nfrequency_decimation = 16\nclutter_length = 0\nmin_acceleration = -300.0\nmax_acceleration = 300.0\nnum_cohints_per_file = 100\nnode_gpus = 0\ndpt_ipp_delay_parameter = 5\n'
FALSE_TRACK_BUDGET_PER_24H = 1.0
RANDOM_SEED = 20260820
HOURS_TO_RUN = None
SWERLING_MAX_BATCHES = None
HDF5_PROGRESS_EVERY = 25
NULL_CHECKPOINT_EVERY = 1
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HOURLY_ROOT.mkdir(parents=True, exist_ok=True)
SWERLING_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_STAGE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_SWERLING_ROOT.mkdir(parents=True, exist_ok=True)
DETECTOR_CONFIGS = [{'config_id': f'fixed_{value:g}db', 'method': 'Fixed SNR', 'kind': 'fixed', 'threshold': float(value), 'units': 'dB'} for value in FIXED_THRESHOLDS_DB] + [{'config_id': f'ca_{value:g}', 'method': 'CA-CFAR', 'kind': 'ca', 'threshold': float(value), 'units': 'CUT/local mean'} for value in CFAR_THRESHOLDS] + [{'config_id': f'os_{value:g}', 'method': 'OS-CFAR', 'kind': 'os', 'threshold': float(value), 'units': 'CUT/order statistic'} for value in CFAR_THRESHOLDS]
CONFIG_PAYLOAD = {'cohint_sec': COHINT_SEC, 'snr_smooth_window': SNR_SMOOTH_WINDOW, 'training_cells_each_side': TRAIN_CELLS, 'guard_cells_each_side': GUARD_CELLS, 'os_order_fraction': OS_ORDER_FRACTION, 'detector_configs': DETECTOR_CONFIGS, 'track_rules': TRACK_RULES, 'max_link_gap_steps': MAX_LINK_GAP_STEPS, 'max_abs_radial_speed_km_s': MAX_ABS_RADIAL_SPEED_KM_S, 'link_range_padding_km': LINK_RANGE_PADDING_KM, 'max_track_rmse_km': MAX_TRACK_RMSE_KM, 'null_trials_per_hour': NULL_TRIALS_PER_HOUR, 'null_block_steps': NULL_BLOCK_STEPS, 'hardtarget_commit': HARDTARGET_COMMIT, 'n_ipp': N_IPP, 'ipp_seconds': IPP_SEC, 'swerling_models': SWERLING_MODELS, 'swerling_output_snr_db': SWERLING_OUTPUT_SNR_DB, 'swerling_durations_sec': SWERLING_DURATIONS_SEC, 'swerling_replicates_per_cell': SWERLING_REPLICATES_PER_CELL, 'swerling_batch_size': SWERLING_BATCH_SIZE, 'swerling_quiet_gap_sec': SWERLING_QUIET_GAP_SEC, 'swerling_phase_model': SWERLING_PHASE_MODEL, 'swerling_experiment': SWERLING_EXPERIMENT, 'swerling_gmf_config': SWERLING_GMF_CONFIG_TEXT, 'random_seed': RANDOM_SEED}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG_PAYLOAD, sort_keys=True).encode('utf-8')).hexdigest()[:16]
print('=' * 78)
print('research RAW SWERLING + 24-HOUR DETECTOR BENCHMARK', flush=True)
print('=' * 78)
print('Configuration hash:', CONFIG_HASH)
print('Output root:', OUTPUT_ROOT)
print('Detector configurations:', len(DETECTOR_CONFIGS))
print('Track rules:', [rule['track_rule'] for rule in TRACK_RULES])
print('Null trials per hour:', NULL_TRIALS_PER_HOUR)
print('Fast continuation target/workers:', NULL_TRIALS_TARGET_PER_HOUR, NULL_WORKERS)
print('Pulses per coherent integration:', N_IPP)
print('Raw Swerling trials:', len(SWERLING_MODELS) * len(SWERLING_OUTPUT_SNR_DB) * len(SWERLING_DURATIONS_SEC) * SWERLING_REPLICATES_PER_CELL)


In [ ]:
LOG_PATH = OUTPUT_ROOT / "live_progress.log"


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def log(message, hour_key="campaign", step="status"):
    line = f"[{utc_now()} | {hour_key} | {step}] {message}"
    print(line, flush=True)
    with open(LOG_PATH, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")
        handle.flush()


def atomic_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")

    def json_default(value):
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, Path):
            return str(value)
        raise TypeError(f"Cannot JSON-serialize {type(value).__name__}")

    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            sort_keys=True,
            default=json_default,
        )
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    compression = "gzip" if path.name.endswith(".gz") else None
    frame.to_csv(temporary, index=False, compression=compression)
    os.replace(temporary, path)


def seed_for(*parts):
    payload = "|".join(str(part) for part in parts)
    digest = hashlib.sha256(payload.encode("utf-8")).digest()
    return (RANDOM_SEED + int.from_bytes(digest[:8], "little")) % (2**32)


def memory_status():
    try:
        import psutil

        process_gib = psutil.Process().memory_info().rss / 2**30
        available_gib = psutil.virtual_memory().available / 2**30
        return f"process={process_gib:.2f} GiB, available={available_gib:.2f} GiB"
    except Exception:
        return "memory status unavailable"


def source_for_hour(hour_key):
    if hour_key == "20180104_12":
        return LEGACY_HOUR_DIR, "legacy"
    return PRODUCTION_ROOT / hour_key, "production"


def campaign_hours():
    expected = [
        (pd.Timestamp("2018-01-04 12:00:00", tz="UTC") + pd.Timedelta(hours=index))
        .strftime("%Y%m%d_%H")
        for index in range(24)
    ]

    rows = []
    for hour_key in expected:
        source, layout = source_for_hour(hour_key)
        marker = source / "COMPLETE.json"
        h5_count = len(list(source.rglob("gmf-*.h5"))) if source.exists() else 0
        valid = bool(source.exists() and h5_count > 0)
        if layout == "production":
            valid = valid and marker.is_file()
        rows.append(
            {
                "hour_key": hour_key,
                "layout": layout,
                "source": str(source),
                "source_exists": source.exists(),
                "gmf_hdf5_files": h5_count,
                "source_valid": valid,
            }
        )

    inventory = pd.DataFrame(rows)
    if not inventory["source_valid"].all():
        display(inventory)
        bad = inventory.loc[~inventory.source_valid, "hour_key"].tolist()
        raise RuntimeError(f"GMF inventory is incomplete for: {bad}")
    return inventory


manifest_path = OUTPUT_ROOT / "run_manifest.json"
if manifest_path.exists():
    existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if existing_manifest.get("config_hash") != CONFIG_HASH:
        raise RuntimeError(
            "Output directory contains a different benchmark configuration. "
            "Change OUTPUT_ROOT before running the new configuration."
        )
else:
    atomic_json(
        {
            "created_at_utc": utc_now(),
            "config_hash": CONFIG_HASH,
            "configuration": CONFIG_PAYLOAD,
            "calibration_design": (
                "Target recovery uses complex DigitalRF echoes before the pinned "
                "Hardtarget GMF. Real 24-hour data supply false-track controls. "
                "RCS/diameter conversion remains a separate radar-equation analysis."
            ),
        },
        manifest_path,
    )


inventory = campaign_hours()
display(inventory)
log(
    f"Inventory passed: {len(inventory)}/24 hours, "
    f"{inventory.gmf_hdf5_files.sum()} HDF5 files total."
)


In [ ]:
def safe_remove_local(path):
    path = Path(path).resolve()
    allowed = LOCAL_STAGE_ROOT.resolve()
    if path == allowed or allowed not in path.parents:
        raise RuntimeError(f"Refusing unsafe local deletion: {path}")
    if path.exists():
        shutil.rmtree(path)


def stage_hour(hour_key):
    source, layout = source_for_hour(hour_key)
    destination = LOCAL_STAGE_ROOT / hour_key
    safe_remove_local(destination)
    destination.mkdir(parents=True, exist_ok=False)

    log(
        f"Staging {source} to local storage; source has "
        f"{len(list(source.rglob('gmf-*.h5')))} HDF5 files.",
        hour_key,
        "stage",
    )
    started = time.monotonic()
    subprocess.run(
        ["rsync", "-a", "--info=progress2", f"{source}/", f"{destination}/"],
        check=True,
    )
    elapsed = time.monotonic() - started
    local_files = sorted(destination.rglob("gmf-*.h5"))
    if not local_files:
        raise RuntimeError(f"No staged HDF5 files for {hour_key}")
    log(
        f"Staging complete in {elapsed / 60:.1f} min: "
        f"{len(local_files)} HDF5 files.",
        hour_key,
        "stage",
    )
    return destination, local_files, layout


def valid_odd_window(requested, length):
    maximum = length if length % 2 else length - 1
    value = min(requested, maximum)
    if value % 2 == 0:
        value -= 1
    return max(value, 3)


def load_hour(hour_key, files):
    gmf_blocks = []
    noise_vectors = []
    time_blocks = []
    ranges_total_m = None

    started = time.monotonic()
    for number, file_path in enumerate(files, start=1):
        if number == 1 or number % HDF5_PROGRESS_EVERY == 0 or number == len(files):
            elapsed = time.monotonic() - started
            rate = number / max(elapsed, 1e-9)
            remaining = (len(files) - number) / max(rate, 1e-9)
            log(
                f"HDF5 {number}/{len(files)}; elapsed={elapsed / 60:.1f} min; "
                f"ETA={remaining / 60:.1f} min; {memory_status()}",
                hour_key,
                "load",
            )

        with h5py.File(file_path, "r") as handle:
            required = ["gmf", "gmf_zero_frequency", "integration_index", "ranges"]
            missing = [name for name in required if name not in handle]
            if missing:
                raise KeyError(f"{file_path.name} missing datasets: {missing}")

            gmf_file = np.asarray(handle["gmf"][()], dtype=np.float32)
            zero_file = np.asarray(handle["gmf_zero_frequency"][()], dtype=np.float32)
            integration_index = np.asarray(handle["integration_index"][()], dtype=float)
            ranges_file = np.asarray(handle["ranges"][()], dtype=float)

            if gmf_file.shape != zero_file.shape:
                raise ValueError(f"GMF/noise mismatch in {file_path.name}")
            if gmf_file.shape[0] != len(integration_index):
                raise ValueError(f"Time/GMF row mismatch in {file_path.name}")

            if ranges_total_m is None:
                ranges_total_m = ranges_file
            elif not np.allclose(ranges_total_m, ranges_file):
                raise ValueError(f"Range grid changed in {file_path.name}")

            if "epoch_unix" in handle:
                file_start = float(np.asarray(handle["epoch_unix"][()]).reshape(-1)[0])
            else:
                match = re.search(r"gmf-(\d+)\.h5$", file_path.name)
                if not match:
                    raise RuntimeError(f"Cannot parse start time from {file_path.name}")
                file_start = int(match.group(1)) / 1e6

            gmf_blocks.append(gmf_file)
            noise_vectors.append(np.nanmedian(zero_file, axis=0))
            time_blocks.append(file_start + integration_index * COHINT_SEC)

    gmf = np.concatenate(gmf_blocks, axis=0)
    time_unix = np.concatenate(time_blocks)
    noise_vectors = np.stack(noise_vectors)
    del gmf_blocks, time_blocks

    order = np.argsort(time_unix)
    gmf = gmf[order]
    time_unix = time_unix[order]

    ranges_slant_km = np.asarray(ranges_total_m, dtype=float) / 2000.0
    noise_floor = np.nanmedian(noise_vectors, axis=0)
    window = valid_odd_window(SNR_SMOOTH_WINDOW, len(noise_floor))
    noise_floor_smooth = savgol_filter(
        noise_floor,
        window_length=window,
        polyorder=1,
        mode="nearest",
    )
    noise_floor_smooth = np.maximum(noise_floor_smooth, EPS).astype(np.float32)

    elapsed = time.monotonic() - started
    log(
        f"Loaded shape={gmf.shape}, duration="
        f"{(time_unix[-1] - time_unix[0] + COHINT_SEC) / 3600:.4f} h, "
        f"physical range={ranges_slant_km.min():.3f}-{ranges_slant_km.max():.3f} km, "
        f"elapsed={elapsed / 60:.1f} min.",
        hour_key,
        "load",
    )
    return {
        "gmf": gmf,
        "time_unix": time_unix,
        "time_sec": time_unix - time_unix[0],
        "ranges_slant_km": ranges_slant_km,
        "noise_floor": noise_floor_smooth,
        "source_noise_floor_unsmoothed": noise_floor,
        "observation_hours": (
            time_unix[-1] - time_unix[0] + COHINT_SEC
        ) / 3600.0,
        "hdf5_files": len(files),
    }


def derive_snr(gmf, noise_floor):
    sqrt_gmf = np.sqrt(np.maximum(gmf, 0.0), dtype=np.float32)
    sqrt_noise = np.sqrt(noise_floor, dtype=np.float32)
    snr_linear = (
        (sqrt_gmf - sqrt_noise[None, :]) ** 2
        / np.maximum(noise_floor[None, :], EPS)
    ).astype(np.float32)
    snr_db = (
        10.0 * np.log10(np.maximum(snr_linear, 1e-12))
    ).astype(np.float32)
    return snr_linear, snr_db


def ca_cfar_ratio(x, train=TRAIN_CELLS, guard=GUARD_CELLS):
    x = np.asarray(x, dtype=np.float32)
    number_times, number_ranges = x.shape
    margin = train + guard
    if 2 * margin + 1 >= number_ranges:
        raise ValueError("CA-CFAR window is too large")

    ratio = np.full(x.shape, np.nan, dtype=np.float32)
    cumulative = np.empty((number_times, number_ranges + 1), dtype=np.float64)
    cumulative[:, 0] = 0.0
    np.cumsum(x, axis=1, dtype=np.float64, out=cumulative[:, 1:])
    indices = np.arange(margin, number_ranges - margin)
    left = (
        cumulative[:, indices - guard]
        - cumulative[:, indices - guard - train]
    )
    right = (
        cumulative[:, indices + guard + train + 1]
        - cumulative[:, indices + guard + 1]
    )
    reference = (left + right) / (2.0 * train)
    ratio[:, indices] = x[:, indices] / np.maximum(reference, EPS)
    return ratio


def os_cfar_ratio(
    x,
    train=TRAIN_CELLS,
    guard=GUARD_CELLS,
    order_fraction=OS_ORDER_FRACTION,
    block_rows=128,
):
    x = np.asarray(x, dtype=np.float32)
    number_times, number_ranges = x.shape
    margin = train + guard
    width = 2 * margin + 1
    if width >= number_ranges:
        raise ValueError("OS-CFAR window is too large")

    ratio = np.full(x.shape, np.nan, dtype=np.float32)
    number_reference = 2 * train
    kth = int(np.ceil(order_fraction * number_reference)) - 1

    for start in range(0, number_times, block_rows):
        stop = min(start + block_rows, number_times)
        block = x[start:stop]
        windows = sliding_window_view(block, window_shape=width, axis=1)
        references = np.concatenate(
            (windows[..., :train], windows[..., -train:]),
            axis=-1,
        )
        order_statistic = np.partition(references, kth, axis=-1)[..., kth]
        cuts = block[:, margin:number_ranges - margin]
        ratio[start:stop, margin:number_ranges - margin] = (
            cuts / np.maximum(order_statistic, EPS)
        )
    return ratio


def calculate_score_fields(hour_key, data):
    started = time.monotonic()
    log("Calculating derived SNR field.", hour_key, "scores")
    snr_linear, snr_db = derive_snr(data["gmf"], data["noise_floor"])
    log("Calculating CA-CFAR field.", hour_key, "scores")
    ca_score = ca_cfar_ratio(data["gmf"])
    log("Calculating OS-CFAR field.", hour_key, "scores")
    os_score = os_cfar_ratio(data["gmf"])
    log("Calculating range-local maxima.", hour_key, "scores")
    local_maximum = (
        data["gmf"]
        == maximum_filter1d(data["gmf"], size=3, axis=1, mode="nearest")
    )
    valid_range = np.zeros(data["gmf"].shape[1], dtype=bool)
    margin = TRAIN_CELLS + GUARD_CELLS
    valid_range[margin:-margin] = True
    log(
        f"All score fields complete in {(time.monotonic() - started) / 60:.1f} min; "
        f"CFAR-valid range={data['ranges_slant_km'][valid_range].min():.3f}-"
        f"{data['ranges_slant_km'][valid_range].max():.3f} km; {memory_status()}",
        hour_key,
        "scores",
    )
    return {
        "fixed": snr_db,
        "ca": ca_score,
        "os": os_score,
        "snr_linear": snr_linear,
        "local_maximum": local_maximum,
        "valid_range": valid_range,
    }


In [ ]:
class UnionFind:
    def __init__(self, size):
        self.parent = np.arange(size, dtype=np.int64)
        self.rank = np.zeros(size, dtype=np.int8)

    def find(self, value):
        value = int(value)
        while self.parent[value] != value:
            self.parent[value] = self.parent[self.parent[value]]
            value = int(self.parent[value])
        return value

    def union(self, first, second):
        root_first = self.find(first)
        root_second = self.find(second)
        if root_first == root_second:
            return
        if self.rank[root_first] < self.rank[root_second]:
            root_first, root_second = root_second, root_first
        self.parent[root_second] = root_first
        if self.rank[root_first] == self.rank[root_second]:
            self.rank[root_first] += 1


def fit_component(component_ids, time_index, range_index, score, time_sec, ranges_km):
    component_ids = np.asarray(component_ids, dtype=int)
    if component_ids.size < 3:
        return None

                                                                   
                                                                             
    selected = []
    component_times = time_index[component_ids]
    for value in np.unique(component_times):
        members = component_ids[component_times == value]
        selected.append(int(members[np.nanargmax(score[members])]))
    selected = np.asarray(selected, dtype=int)
    selected = selected[np.argsort(time_index[selected])]
    if selected.size < 3:
        return None

    keep = np.ones(selected.size, dtype=bool)
    for _ in range(3):
        ids = selected[keep]
        if ids.size < 3:
            return None
        times = time_sec[time_index[ids]]
        ranges = ranges_km[range_index[ids]]
        center = float(np.median(times))
        relative = times - center
        duration = float(times.max() - times.min())
        degree = 2 if ids.size >= 5 and duration >= 1.0 else 1
        coefficients = np.polyfit(relative, ranges, degree)
        residuals = ranges - np.polyval(coefficients, relative)
        residual_median = np.median(residuals)
        residual_mad = np.median(np.abs(residuals - residual_median))
        robust_scale = max(1.4826 * residual_mad, 1e-6)
        limit = max(MAX_TRACK_RMSE_KM, 3.0 * robust_scale)
        new_local_keep = np.abs(residuals - residual_median) <= limit
        if np.all(new_local_keep):
            break
        new_keep = np.zeros_like(keep)
        new_keep[np.where(keep)[0][new_local_keep]] = True
        keep = new_keep

    ids = selected[keep]
    if ids.size < 3:
        return None
    times = time_sec[time_index[ids]]
    ranges = ranges_km[range_index[ids]]
    center = float(np.median(times))
    relative = times - center
    duration = float(times.max() - times.min())
    degree = 2 if ids.size >= 5 and duration >= 1.0 else 1
    coefficients = np.polyfit(relative, ranges, degree)
    fitted = np.polyval(coefficients, relative)
    residuals = ranges - fitted
    rmse = float(np.sqrt(np.mean(residuals**2)))

    if degree == 2:
        velocity_center = float(coefficients[1])
        acceleration = float(2.0 * coefficients[0])
        endpoint_velocities = 2.0 * coefficients[0] * np.array(
            [times.min() - center, times.max() - center]
        ) + coefficients[1]
    else:
        velocity_center = float(coefficients[0])
        acceleration = 0.0
        endpoint_velocities = np.array([coefficients[0], coefficients[0]])

    return {
        "candidate_ids": component_ids,
        "selected_ids": ids,
        "number_unique_times": int(ids.size),
        "start_sec": float(times.min()),
        "end_sec": float(times.max()),
        "duration_sec": duration,
        "center_sec": center,
        "degree": degree,
        "coefficients": coefficients,
        "trajectory_rmse_km": rmse,
        "velocity_center_km_s": velocity_center,
        "acceleration_km_s2": acceleration,
        "maximum_endpoint_velocity_km_s": float(np.max(np.abs(endpoint_velocities))),
        "peak_detector_score": float(np.nanmax(score[component_ids])),
        "peak_snr_or_score": float(np.nanmax(score[component_ids])),
        "time_indices": time_index[ids],
        "range_indices": range_index[ids],
    }


def build_tracks(candidate_mask, detector_score, time_sec, ranges_km):
    time_index, range_index = np.where(candidate_mask)
    number_points = len(time_index)
    if number_points == 0:
        return [], 0

    point_scores = detector_score[time_index, range_index]
    spacing_km = float(np.median(np.diff(ranges_km)))
    union = UnionFind(number_points)

    by_time = {}
    for value in np.unique(time_index):
        ids = np.where(time_index == value)[0]
        order = np.argsort(range_index[ids])
        by_time[int(value)] = ids[order]

    for current_time, current_ids in by_time.items():
        current_ranges = range_index[current_ids]
        for gap_steps in range(1, MAX_LINK_GAP_STEPS + 1):
            future_time = current_time + gap_steps
            future_ids = by_time.get(future_time)
            if future_ids is None:
                continue
            future_ranges = range_index[future_ids]
            delta_t = gap_steps * COHINT_SEC
            tolerance_km = (
                MAX_ABS_RADIAL_SPEED_KM_S * delta_t + LINK_RANGE_PADDING_KM
            )
            tolerance_bins = int(math.ceil(tolerance_km / spacing_km))
            for point_id, range_bin in zip(current_ids, current_ranges):
                left = np.searchsorted(future_ranges, range_bin - tolerance_bins, side="left")
                right = np.searchsorted(future_ranges, range_bin + tolerance_bins, side="right")
                for future_id in future_ids[left:right]:
                    union.union(point_id, int(future_id))

    components = defaultdict(list)
    for point_id in range(number_points):
        components[union.find(point_id)].append(point_id)

    tracks = []
    for component in components.values():
        fitted = fit_component(
            component,
            time_index,
            range_index,
            point_scores,
            time_sec,
            ranges_km,
        )
        if fitted is not None:
            tracks.append(fitted)
    return tracks, number_points


def accepted_tracks(tracks, rule):
    return [
        track
        for track in tracks
        if track["number_unique_times"] >= rule["min_unique_times"]
        and track["duration_sec"] + 1e-9 >= rule["min_duration_sec"]
        and track["trajectory_rmse_km"] <= MAX_TRACK_RMSE_KM
        and track["maximum_endpoint_velocity_km_s"] <= MAX_ABS_RADIAL_SPEED_KM_S
    ]


def candidate_mask_for(config, scores):
    return (
        (scores[config["kind"]] >= config["threshold"])
        & scores["local_maximum"]
        & scores["valid_range"][None, :]
    )


def track_row(hour_key, config, rule, track_number, track, data):
    ids_time = track["time_indices"]
    ids_range = track["range_indices"]
    return {
        "hour_key": hour_key,
        "config_id": config["config_id"],
        "method": config["method"],
        "threshold": config["threshold"],
        "units": config["units"],
        "track_rule": rule["track_rule"],
        "track_number": int(track_number),
        "start_time_unix": float(data["time_unix"][ids_time].min()),
        "end_time_unix": float(data["time_unix"][ids_time].max()),
        "duration_sec": track["duration_sec"],
        "number_unique_times": track["number_unique_times"],
        "minimum_range_km": float(data["ranges_slant_km"][ids_range].min()),
        "maximum_range_km": float(data["ranges_slant_km"][ids_range].max()),
        "reference_range_km": float(np.median(data["ranges_slant_km"][ids_range])),
        "velocity_center_km_s": track["velocity_center_km_s"],
        "acceleration_km_s2": track["acceleration_km_s2"],
        "trajectory_rmse_km": track["trajectory_rmse_km"],
        "maximum_endpoint_velocity_km_s": track["maximum_endpoint_velocity_km_s"],
        "peak_detector_score": track["peak_detector_score"],
    }


In [ ]:
def run_real_sweep(hour_key, data, scores, hour_out):
    summary_rows = []
    track_rows = []
    started = time.monotonic()

    for config_number, config in enumerate(DETECTOR_CONFIGS, start=1):
        mask = candidate_mask_for(config, scores)
        tracks, candidate_cells = build_tracks(
            mask,
            scores[config["kind"]],
            data["time_sec"],
            data["ranges_slant_km"],
        )
        for rule in TRACK_RULES:
            accepted = accepted_tracks(tracks, rule)
            summary_rows.append(
                {
                    "hour_key": hour_key,
                    "config_id": config["config_id"],
                    "method": config["method"],
                    "threshold": config["threshold"],
                    "units": config["units"],
                    "track_rule": rule["track_rule"],
                    "candidate_cells": candidate_cells,
                    "fitted_components": len(tracks),
                    "physical_tracks": len(accepted),
                    "observation_hours": data["observation_hours"],
                }
            )
            for track_number, track in enumerate(accepted, start=1):
                track_rows.append(
                    track_row(
                        hour_key,
                        config,
                        rule,
                        track_number,
                        track,
                        data,
                    )
                )

        log(
            f"Real sweep config {config_number}/{len(DETECTOR_CONFIGS)}: "
            f"{config['config_id']}, candidates={candidate_cells}, components={len(tracks)}",
            hour_key,
            "real",
        )

    summary = pd.DataFrame(summary_rows)
    tracks = pd.DataFrame(track_rows)
    atomic_csv(summary, hour_out / "real_summary.csv")
    atomic_csv(tracks, hour_out / "real_tracks.csv.gz")
    atomic_json(
        {
            "status": "complete",
            "completed_at_utc": utc_now(),
            "config_hash": CONFIG_HASH,
            "summary_rows": len(summary),
            "track_rows": len(tracks),
            "elapsed_seconds": time.monotonic() - started,
        },
        hour_out / "REAL_COMPLETE.json",
    )
    return summary, tracks


def block_permutation_indices(number_rows, block_steps, rng):
    blocks = [
        np.arange(start, min(start + block_steps, number_rows), dtype=int)
        for start in range(0, number_rows, block_steps)
    ]
    order = rng.permutation(len(blocks))
    return np.concatenate([blocks[index] for index in order])


_NULL_WORKER_CONTEXT = None


def _run_one_null_trial(trial_id):
    context = _NULL_WORKER_CONTEXT
    if context is None:
        raise RuntimeError("Null worker context was not initialized")

    hour_key = context["hour_key"]
    data = context["data"]
    scores = context["scores"]
    rng = np.random.default_rng(seed_for(hour_key, "null", trial_id))
    permutation = block_permutation_indices(
        len(data["time_sec"]), NULL_BLOCK_STEPS, rng
    )
    permuted_local_maximum = scores["local_maximum"][permutation]
    rows = []

    for config in DETECTOR_CONFIGS:
        permuted_score = scores[config["kind"]][permutation]
        mask = (
            (permuted_score >= config["threshold"])
            & permuted_local_maximum
            & scores["valid_range"][None, :]
        )
        tracks, candidate_cells = build_tracks(
            mask,
            permuted_score,
            data["time_sec"],
            data["ranges_slant_km"],
        )
        for rule in TRACK_RULES:
            rows.append(
                {
                    "hour_key": hour_key,
                    "trial_id": trial_id,
                    "config_id": config["config_id"],
                    "method": config["method"],
                    "threshold": config["threshold"],
                    "units": config["units"],
                    "track_rule": rule["track_rule"],
                    "candidate_cells": candidate_cells,
                    "false_tracks": len(accepted_tracks(tracks, rule)),
                    "null_exposure_hours": data["observation_hours"],
                    "block_steps": NULL_BLOCK_STEPS,
                    "block_seconds": NULL_BLOCK_STEPS * COHINT_SEC,
                }
            )
    return trial_id, rows


def run_null_controls(hour_key, data, scores, hour_out):
    global _NULL_WORKER_CONTEXT

    output_path = hour_out / "null_trials.csv.gz"
    if output_path.exists():
        frame = pd.read_csv(output_path)
    else:
        frame = pd.DataFrame()

    expected_rows_per_trial = len(DETECTOR_CONFIGS) * len(TRACK_RULES)
    completed = set()
    if len(frame):
        counts = frame.groupby("trial_id").size()
        completed = set(
            counts[counts == expected_rows_per_trial].index.astype(int)
        )
        frame = frame[frame.trial_id.isin(completed)].copy()

    requested_ids = list(range(NULL_TRIALS_TARGET_PER_HOUR))
    pending = [trial_id for trial_id in requested_ids if trial_id not in completed]
    started = time.monotonic()

    def checkpoint_trial(trial_id, rows):
        nonlocal frame
        frame = pd.concat([frame, pd.DataFrame(rows)], ignore_index=True)
        completed.add(int(trial_id))
        atomic_csv(frame, output_path)
        log(
            f"Fast null trial {trial_id + 1}/{NULL_TRIALS_TARGET_PER_HOUR} "
            f"complete; {len(completed)} total saved; "
            f"elapsed={(time.monotonic() - started) / 60:.1f} min",
            hour_key,
            "null",
        )

    _NULL_WORKER_CONTEXT = {
        "hour_key": hour_key,
        "data": data,
        "scores": scores,
    }

                                                                    
                                                                
    if len(pending) > 1 and NULL_WORKERS > 1:
        worker_count = min(NULL_WORKERS, len(pending))
        log(
            f"Running {len(pending)} pending null trials with "
            f"{worker_count} forked CPU workers.",
            hour_key,
            "null",
        )
        try:
            fork_context = mp.get_context("fork")
            with ProcessPoolExecutor(
                max_workers=worker_count,
                mp_context=fork_context,
            ) as executor:
                futures = {
                    executor.submit(_run_one_null_trial, trial_id): trial_id
                    for trial_id in pending
                }
                for future in as_completed(futures):
                    trial_id, rows = future.result()
                    checkpoint_trial(trial_id, rows)
        except Exception as error:
            log(
                f"Parallel null execution failed ({type(error).__name__}: "
                f"{error}); finishing missing trials sequentially.",
                hour_key,
                "null",
            )

                                                                      
    missing = [trial_id for trial_id in requested_ids if trial_id not in completed]
    for trial_id in missing:
        completed_trial, rows = _run_one_null_trial(trial_id)
        checkpoint_trial(completed_trial, rows)

    _NULL_WORKER_CONTEXT = None
    completed_count = int(frame.trial_id.nunique()) if len(frame) else 0
    atomic_json(
        {
            "status": "complete",
            "completed_at_utc": utc_now(),
            "config_hash": CONFIG_HASH,
            "original_exhaustive_trials_per_hour": NULL_TRIALS_PER_HOUR,
            "fast_target_trials_per_hour": NULL_TRIALS_TARGET_PER_HOUR,
            "completed_null_trials": completed_count,
            "workers": NULL_WORKERS,
            "rows": len(frame),
            "execution_note": (
                "Fast continuation retains every already completed trial "
                "and targets four trials per remaining real hour. Ranking "
                "uses recorded null_exposure_hours, so unequal completed "
                "trial counts are handled explicitly."
            ),
        },
        hour_out / "NULL_COMPLETE.json",
    )
    return frame


In [ ]:
SWERLING_EPOCH = pd.Timestamp("2018-01-04T12:00:00Z")
SWERLING_EPOCH_UNIX = float(SWERLING_EPOCH.timestamp())


def normalized_swerling_rcs(model, size, rng):
    if model == "Swerling0":
        return np.ones(size, dtype=np.float64)
    if model in {"Swerling1", "Swerling2"}:
        return rng.exponential(scale=1.0, size=size)
    if model in {"Swerling3", "Swerling4"}:
        return rng.gamma(shape=2.0, scale=0.5, size=size)
    raise ValueError(f"Unknown Swerling model: {model}")


def swerling_update_rate(model):
    if model == "Swerling0":
        return "nonfluctuating"
    if model in {"Swerling1", "Swerling3"}:
        return "scan-to-scan"
    if model in {"Swerling2", "Swerling4"}:
        return "pulse-to-pulse"
    raise ValueError(model)


def validate_swerling_statistics():
    rng = np.random.default_rng(seed_for("swerling_distribution_qa"))
    rows = []
    for model in SWERLING_MODELS:
        sample = normalized_swerling_rcs(model, 200_000, rng)
        expected_variance = {
            "Swerling0": 0.0,
            "Swerling1": 1.0,
            "Swerling2": 1.0,
            "Swerling3": 0.5,
            "Swerling4": 0.5,
        }[model]
        if model == "Swerling0":
            ks_statistic = 0.0
        elif model in {"Swerling1", "Swerling2"}:
            ks_statistic = float(kstest(sample, expon(scale=1.0).cdf).statistic)
        else:
            ks_statistic = float(
                kstest(sample, gamma_distribution(a=2.0, scale=0.5).cdf).statistic
            )
        rows.append(
            {
                "model": model,
                "update_rate": swerling_update_rate(model),
                "sample_mean": float(sample.mean()),
                "sample_variance": float(sample.var()),
                "expected_variance": expected_variance,
                "ks_statistic": ks_statistic,
            }
        )
    qa = pd.DataFrame(rows)
    if not np.all(np.abs(qa.sample_mean - 1.0) < 0.015):
        raise AssertionError("Swerling mean-RCS validation failed")
    if not np.all(np.abs(qa.sample_variance - qa.expected_variance) < 0.04):
        raise AssertionError("Swerling RCS variance validation failed")
    if not np.all(qa.ks_statistic < 0.01):
        raise AssertionError("Swerling distribution KS validation failed")
    atomic_csv(qa, SWERLING_ROOT / "swerling_distribution_qa.csv")
    return qa


def build_swerling_trial_manifest():
    path = SWERLING_ROOT / "swerling_trial_manifest.csv"
    marker = SWERLING_ROOT / "TRIAL_MANIFEST.json"
    if path.exists() and marker.exists():
        payload = json.loads(marker.read_text(encoding="utf-8"))
        if payload.get("config_hash") == CONFIG_HASH:
            return pd.read_csv(path)

    specifications = []
    for model in SWERLING_MODELS:
        for duration_sec in SWERLING_DURATIONS_SEC:
            for requested_output_snr_db in SWERLING_OUTPUT_SNR_DB:
                for replicate in range(SWERLING_REPLICATES_PER_CELL):
                    specifications.append(
                        {
                            "model": model,
                            "update_rate": swerling_update_rate(model),
                            "duration_sec": float(duration_sec),
                            "requested_output_snr_db": float(requested_output_snr_db),
                            "replicate": int(replicate),
                        }
                    )

    shuffle_rng = np.random.default_rng(seed_for("swerling_manifest_shuffle"))
    order = shuffle_rng.permutation(len(specifications))
    rows = []
    for trial_id, source_index in enumerate(order):
        row = dict(specifications[int(source_index)])
        trial_rng = np.random.default_rng(seed_for("swerling_trial", trial_id))
        row.update(
            {
                "trial_id": int(trial_id),
                "fold": int(trial_id % 6),
                "batch_id": int(trial_id // SWERLING_BATCH_SIZE),
                "batch_position": int(trial_id % SWERLING_BATCH_SIZE),
                "center_range_km": float(trial_rng.uniform(1045.0, 1065.0)),
                "radial_speed_km_s": float(
                    trial_rng.choice(SWERLING_SPEED_CHOICES_KM_S)
                ),
                "radial_acceleration_km_s2": float(
                    trial_rng.choice(SWERLING_ACCELERATION_CHOICES_KM_S2)
                ),
            }
        )
        rows.append(row)

    manifest = pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)
    atomic_csv(manifest, path)
    atomic_json(
        {
            "status": "complete",
            "created_at_utc": utc_now(),
            "config_hash": CONFIG_HASH,
            "trials": len(manifest),
            "batches": int(manifest.batch_id.nunique()),
        },
        marker,
    )
    return manifest


def add_batch_timing(batch):
    batch = batch.sort_values("batch_position").copy().reset_index(drop=True)
    cursor = SWERLING_BATCH_EDGE_PAD_SEC
    starts = []
    stops = []
    active_durations = []
    for row in batch.itertuples():
        start = math.ceil((cursor - 1e-12) / COHINT_SEC) * COHINT_SEC
        active_duration = float(row.duration_sec) + COHINT_SEC
        stop = start + active_duration
        starts.append(start)
        stops.append(stop)
        active_durations.append(active_duration)
        cursor = stop + SWERLING_QUIET_GAP_SEC
    batch["start_sec"] = starts
    batch["stop_sec"] = stops
    batch["active_duration_sec"] = active_durations
    return batch, cursor + SWERLING_BATCH_EDGE_PAD_SEC


def build_pulse_profile(batch, total_duration_sec):
    number_pulses = int(math.ceil(total_duration_sec / IPP_SEC)) + 2
    pulse_snr = np.zeros(number_pulses, dtype=np.float64)
    mean_factors = []
    minimum_factors = []
    maximum_factors = []

    for row in batch.itertuples():
        first = max(0, int(math.ceil((row.start_sec - 1e-10) / IPP_SEC)))
        last = min(number_pulses, int(math.ceil((row.stop_sec - 1e-10) / IPP_SEC)))
        pulse_ids = np.arange(first, last, dtype=int)
        rng = np.random.default_rng(seed_for("swerling_rcs", row.trial_id))
        if row.model in {"Swerling1", "Swerling3"}:
            factors = np.repeat(
                normalized_swerling_rcs(row.model, 1, rng), len(pulse_ids)
            )
        else:
            factors = normalized_swerling_rcs(row.model, len(pulse_ids), rng)
        pulse_snr[pulse_ids] = 10.0 ** (row.input_snr_db / 10.0) * factors
        mean_factors.append(float(np.mean(factors)))
        minimum_factors.append(float(np.min(factors)))
        maximum_factors.append(float(np.max(factors)))

    batch = batch.copy()
    batch["realized_mean_rcs_factor"] = mean_factors
    batch["realized_minimum_rcs_factor"] = minimum_factors
    batch["realized_maximum_rcs_factor"] = maximum_factors
    return pulse_snr, batch


def make_raw_target_functions(batch, pulse_snr):
    starts = batch.start_sec.to_numpy(float)
    stops = batch.stop_sec.to_numpy(float)
    centers_t = 0.5 * (starts + stops)
    centers_r = batch.center_range_km.to_numpy(float)
    speeds = batch.radial_speed_km_s.to_numpy(float)
    accelerations = batch.radial_acceleration_km_s2.to_numpy(float)

    def active_trial_indices(values):
        values = np.asarray(values, dtype=float)
        indices = np.searchsorted(starts, values, side="right") - 1
        clipped = np.clip(indices, 0, len(starts) - 1)
        active = (indices >= 0) & (values < stops[clipped])
        return clipped, active

    def range_function(values):
        values = np.asarray(values, dtype=float)
        indices, active = active_trial_indices(values)
        physical_range_km = np.full(values.shape, 1055.0, dtype=np.float64)
        if np.any(active):
            local = values[active] - centers_t[indices[active]]
            physical_range_km[active] = (
                centers_r[indices[active]]
                + speeds[indices[active]] * local
                + 0.5 * accelerations[indices[active]] * local**2
            )
                                                                      
                                   
        return 2.0e3 * physical_range_km

    def snr_function(values):
        values = np.asarray(values, dtype=float)
        pulse_ids = np.floor((values + 1e-10) / IPP_SEC).astype(int)
        pulse_ids = np.clip(pulse_ids, 0, len(pulse_snr) - 1)
        return pulse_snr[pulse_ids]

    return range_function, snr_function


def safe_remove_swerling_local(path):
    path = Path(path).resolve()
    allowed = LOCAL_SWERLING_ROOT.resolve()
    if path == allowed or allowed not in path.parents:
        raise RuntimeError(f"Refusing unsafe Swerling deletion: {path}")
    if path.exists():
        shutil.rmtree(path)


def simulate_raw_batch(batch_id, batch):
    batch, total_duration_sec = add_batch_timing(batch)
    pulse_snr, batch = build_pulse_profile(batch, total_duration_sec)
    range_function, snr_function = make_raw_target_functions(batch, pulse_snr)

    batch_root = LOCAL_SWERLING_ROOT / f"batch_{str(batch_id).replace('-', '_')}"
    safe_remove_swerling_local(batch_root)
    batch_root.mkdir(parents=True, exist_ok=False)
    drf_root = batch_root / "drf"
    gmf_root = batch_root / "gmf"
    config_path = batch_root / "gmf_config.ini"
    config_path.write_text(SWERLING_GMF_CONFIG_TEXT, encoding="utf-8")

    experiment = dict(SWERLING_EXPERIMENT)
    code_name = experiment.pop("code_name")
    experiment["code"] = hardtarget.load_radar_code(code_name)
    simulation = {
        "epoch": SWERLING_EPOCH.isoformat(),
        "start_time": 0.0,
        "end_time": float(total_duration_sec),
        "target_start_time": 0.0,
        "target_end_time": float(total_duration_sec),
        "noise_sigma": 1.0,
        "tx_amp": 1.0,
    }

    np.random.seed(seed_for("swerling_raw_noise", batch_id))
    log(
        f"Simulating {len(batch)} raw tracks over {total_duration_sec:.1f} s.",
        f"swerling_{batch_id}",
        "raw-voltage",
    )
    hardtarget.simulation.drf(
        str(drf_root),
        range_function,
        simulation,
        experiment,
        chnl="sim",
        snr_function=snr_function,
        dtype=np.complex64,
        clobber=True,
    )
    gmf_root.mkdir(parents=True, exist_ok=True)
    hardtarget.compute_gmf(
        rx=(str(drf_root), "sim"),
        tx=(str(drf_root), "sim"),
        config=str(config_path),
        gmf_method="fgmf",
        gmf_implementation=Impl.c.name,
        clobber=False,
        output=str(gmf_root),
        progress=True,
        subprogress=False,
    )
    files = sorted(gmf_root.rglob("gmf-*.h5"))
    if not files:
        raise RuntimeError(f"Raw Swerling batch {batch_id} produced no GMF files")
    data = load_hour(f"swerling_{batch_id}", files)
    scores = calculate_score_fields(f"swerling_{batch_id}", data)
    return batch_root, batch, data, scores


def expected_trial_range_km(trial, relative_time_sec):
    center_time = 0.5 * (trial.start_sec + trial.stop_sec)
    local = np.asarray(relative_time_sec, dtype=float) - center_time
    return (
        trial.center_range_km
        + trial.radial_speed_km_s * local
        + 0.5 * trial.radial_acceleration_km_s2 * local**2
    )


def raw_trial_recovered(track, rule, trial, data):
    if not accepted_tracks([track], rule):
        return False
    selected_rows = track["time_indices"]
    selected_ranges = data["ranges_slant_km"][track["range_indices"]]
    relative_times = data["time_unix"][selected_rows] - SWERLING_EPOCH_UNIX
    inside = (
        (relative_times >= trial.start_sec - 1e-9)
        & (relative_times <= trial.stop_sec + 1e-9)
    )
    if int(np.sum(inside)) < rule["min_unique_times"]:
        return False
    expected = expected_trial_range_km(trial, relative_times[inside])
    close = np.abs(selected_ranges[inside] - expected) <= max(
        LINK_RANGE_PADDING_KM,
        2.0 * float(np.median(np.diff(data["ranges_slant_km"]))),
    )
    return int(np.sum(close)) >= rule["min_unique_times"]


def measured_trial_output_snr(trial, data, scores):
    relative_times = data["time_unix"] - SWERLING_EPOCH_UNIX
    inside = np.where(
        (relative_times >= trial.start_sec + 0.5 * COHINT_SEC)
        & (relative_times <= trial.stop_sec - 0.5 * COHINT_SEC)
    )[0]
    if not len(inside):
        return np.nan
    expected = expected_trial_range_km(trial, relative_times[inside])
    range_ids = np.array(
        [
            int(np.argmin(np.abs(data["ranges_slant_km"] - value)))
            for value in expected
        ],
        dtype=int,
    )
    return float(np.nanmedian(scores["fixed"][inside, range_ids]))


def evaluate_raw_batch(batch, data, scores, processing_gain_db):
    rows = []
    measured = {
        int(trial.trial_id): measured_trial_output_snr(trial, data, scores)
        for trial in batch.itertuples()
    }
    for config in DETECTOR_CONFIGS:
        mask = candidate_mask_for(config, scores)
        tracks, candidate_cells = build_tracks(
            mask,
            scores[config["kind"]],
            data["time_sec"],
            data["ranges_slant_km"],
        )
        for rule in TRACK_RULES:
            for trial in batch.itertuples():
                recovered = any(
                    raw_trial_recovered(track, rule, trial, data)
                    for track in tracks
                )
                rows.append(
                    {
                        "trial_id": int(trial.trial_id),
                        "fold": int(trial.fold),
                        "batch_id": int(trial.batch_id),
                        "replicate": int(trial.replicate),
                        "model": trial.model,
                        "update_rate": trial.update_rate,
                        "phase_model": SWERLING_PHASE_MODEL,
                        "n_ipp": N_IPP,
                        "duration_sec": float(trial.duration_sec),
                        "requested_output_snr_db": float(
                            trial.requested_output_snr_db
                        ),
                        "input_snr_db": float(trial.input_snr_db),
                        "processing_gain_db": float(processing_gain_db),
                        "median_actual_output_snr_db": measured[int(trial.trial_id)],
                        "realized_mean_rcs_factor": float(
                            trial.realized_mean_rcs_factor
                        ),
                        "realized_minimum_rcs_factor": float(
                            trial.realized_minimum_rcs_factor
                        ),
                        "realized_maximum_rcs_factor": float(
                            trial.realized_maximum_rcs_factor
                        ),
                        "radial_speed_km_s": float(trial.radial_speed_km_s),
                        "radial_acceleration_km_s2": float(
                            trial.radial_acceleration_km_s2
                        ),
                        "config_id": config["config_id"],
                        "method": config["method"],
                        "threshold": config["threshold"],
                        "units": config["units"],
                        "track_rule": rule["track_rule"],
                        "candidate_cells_in_batch": candidate_cells,
                        "recovered": int(recovered),
                    }
                )
    return pd.DataFrame(rows)


def calibrate_processing_gain():
    path = SWERLING_ROOT / "PROCESSING_GAIN_CALIBRATION.json"
    if path.exists():
        payload = json.loads(path.read_text(encoding="utf-8"))
        if payload.get("config_hash") == CONFIG_HASH:
            return float(payload["processing_gain_db"])

    pilot = pd.DataFrame(
        [
            {
                "trial_id": -1,
                "fold": -1,
                "batch_id": -1,
                "batch_position": 0,
                "replicate": 0,
                "model": "Swerling0",
                "update_rate": "nonfluctuating",
                "duration_sec": 8.0,
                "requested_output_snr_db": np.nan,
                "input_snr_db": SWERLING_PILOT_INPUT_SNR_DB,
                "center_range_km": 1055.0,
                "radial_speed_km_s": 0.5,
                "radial_acceleration_km_s2": 0.0,
            }
        ]
    )
    batch_root = None
    try:
        batch_root, timed, data, scores = simulate_raw_batch("pilot", pilot)
        output_snr_db = measured_trial_output_snr(timed.iloc[0], data, scores)
        processing_gain_db = float(output_snr_db - SWERLING_PILOT_INPUT_SNR_DB)
        if not np.isfinite(processing_gain_db):
            raise RuntimeError("Non-finite raw-simulator processing gain")
        atomic_json(
            {
                "status": "complete",
                "completed_at_utc": utc_now(),
                "config_hash": CONFIG_HASH,
                "hardtarget_commit": HARDTARGET_COMMIT,
                "pilot_input_snr_db": SWERLING_PILOT_INPUT_SNR_DB,
                "pilot_median_output_snr_db": output_snr_db,
                "processing_gain_db": processing_gain_db,
            },
            path,
        )
        return processing_gain_db
    finally:
        if batch_root is not None:
            safe_remove_swerling_local(batch_root)


def run_swerling_raw_calibration():
    qa = validate_swerling_statistics()
    display(qa)
    manifest = build_swerling_trial_manifest()
    processing_gain_db = calibrate_processing_gain()
    manifest = manifest.copy()
    manifest["input_snr_db"] = (
        manifest.requested_output_snr_db - processing_gain_db
    )
    atomic_csv(manifest, SWERLING_ROOT / "swerling_trial_manifest_calibrated.csv")

    batch_ids = sorted(manifest.batch_id.unique().astype(int))
    if SWERLING_MAX_BATCHES is not None:
        batch_ids = batch_ids[: int(SWERLING_MAX_BATCHES)]
    batch_output_root = SWERLING_ROOT / "batches"
    batch_output_root.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    statuses = []

    for number, batch_id in enumerate(batch_ids, start=1):
        output_path = batch_output_root / f"batch_{batch_id:04d}.csv.gz"
        marker_path = batch_output_root / f"batch_{batch_id:04d}.json"
        if output_path.exists() and marker_path.exists():
            payload = json.loads(marker_path.read_text(encoding="utf-8"))
            if payload.get("config_hash") == CONFIG_HASH and payload.get("status") == "complete":
                statuses.append({"batch_id": batch_id, "status": "skipped_complete"})
                continue

        batch_root = None
        batch = manifest[manifest.batch_id == batch_id].copy()
        batch_started = time.monotonic()
        try:
            batch_root, timed, data, scores = simulate_raw_batch(batch_id, batch)
            result = evaluate_raw_batch(timed, data, scores, processing_gain_db)
            atomic_csv(result, output_path)
            payload = {
                "status": "complete",
                "completed_at_utc": utc_now(),
                "config_hash": CONFIG_HASH,
                "batch_id": batch_id,
                "trials": len(batch),
                "result_rows": len(result),
                "elapsed_seconds": time.monotonic() - batch_started,
            }
            atomic_json(payload, marker_path)
            statuses.append(payload)
            log(
                f"Raw batch {number}/{len(batch_ids)} complete; "
                f"campaign elapsed={(time.monotonic() - started) / 3600:.2f} h.",
                f"swerling_{batch_id}",
                "complete",
            )
        except Exception as error:
            failure = {
                "status": "failed",
                "failed_at_utc": utc_now(),
                "config_hash": CONFIG_HASH,
                "batch_id": batch_id,
                "error": repr(error),
                "traceback": traceback.format_exc(),
            }
            atomic_json(failure, batch_output_root / f"batch_{batch_id:04d}_FAILURE.json")
            statuses.append(failure)
            log(f"Raw batch failed: {error}", f"swerling_{batch_id}", "failure")
        finally:
            if batch_root is not None:
                safe_remove_swerling_local(batch_root)
            gc.collect()

    status = pd.DataFrame(statuses)
    atomic_csv(status, SWERLING_ROOT / "batch_status.csv")
    expected_batch_ids = sorted(manifest.batch_id.unique().astype(int))
    if SWERLING_MAX_BATCHES is not None:
        preview_paths = [
            batch_output_root / f"batch_{batch_id:04d}.csv.gz"
            for batch_id in batch_ids
        ]
        available = [path for path in preview_paths if path.exists()]
        if not available:
            raise RuntimeError("No quick-test Swerling batch completed")
        preview = pd.concat(
            [pd.read_csv(path) for path in available],
            ignore_index=True,
        )
        atomic_csv(
            preview,
            SWERLING_ROOT / "swerling_trials_quick_test.csv.gz",
        )
        print(
            "QUICK TEST ONLY: remove SWERLING_MAX_BATCHES and rerun for "
            "the full detector calibration."
        )
        return preview

    completed_paths = [
        batch_output_root / f"batch_{batch_id:04d}.csv.gz"
        for batch_id in expected_batch_ids
    ]
    if not all(path.exists() for path in completed_paths):
        missing = [
            path.name for path in completed_paths if not path.exists()
        ]
        raise RuntimeError(
            f"Raw Swerling calibration is incomplete; {len(missing)} batches remain. "
            "Rerun this cell to resume."
        )
    combined = pd.concat(
        [pd.read_csv(path) for path in completed_paths],
        ignore_index=True,
    )
    atomic_csv(combined, SWERLING_ROOT / "swerling_trials_all.csv.gz")
    atomic_json(
        {
            "status": "complete",
            "completed_at_utc": utc_now(),
            "config_hash": CONFIG_HASH,
            "trials": int(combined.trial_id.nunique()),
            "rows": len(combined),
            "processing_gain_db": processing_gain_db,
            "hardtarget_commit": HARDTARGET_COMMIT,
            "phase_model": SWERLING_PHASE_MODEL,
        },
        SWERLING_ROOT / "SWERLING_COMPLETE.json",
    )
    return combined


In [ ]:
def run_self_test():
    rng = np.random.default_rng(12345)
    rows, columns = 120, 160
    noise = np.ones(columns, dtype=np.float32)
    gmf = rng.exponential(1.0, size=(rows, columns)).astype(np.float32)
    start = 30
    for offset in range(15):
        row = start + offset
        column = 75 + offset // 3
        target_linear = 10 ** (20.0 / 10.0)
        gmf[row, column] = (
            math.sqrt(float(gmf[row, column])) + math.sqrt(target_linear)
        ) ** 2
    _, snr_db = derive_snr(gmf, noise)
    local_maximum = gmf == maximum_filter1d(gmf, 3, axis=1, mode="nearest")
    valid = np.zeros(columns, dtype=bool)
    margin = TRAIN_CELLS + GUARD_CELLS
    valid[margin:-margin] = True
    mask = (snr_db >= 10.0) & local_maximum & valid[None, :]
    time_sec = np.arange(rows) * COHINT_SEC
    ranges_km = 1000.0 + np.arange(columns) * 0.15
    tracks, _ = build_tracks(mask, snr_db, time_sec, ranges_km)
    recovered = accepted_tracks(
        tracks,
        {"track_rule": "self_test", "min_unique_times": 6, "min_duration_sec": 1.0},
    )
    if not recovered:
        raise AssertionError("Synthetic-track self-test failed")

    cadence_probe = pd.DataFrame(
        [
            {
                "trial_id": 900001,
                "model": "Swerling1",
                "start_sec": 1.0,
                "stop_sec": 1.4,
                "input_snr_db": 0.0,
            },
            {
                "trial_id": 900002,
                "model": "Swerling2",
                "start_sec": 2.0,
                "stop_sec": 2.4,
                "input_snr_db": 0.0,
            },
        ]
    )
    _, cadence_probe = build_pulse_profile(cadence_probe, 3.0)
    slow = cadence_probe[cadence_probe.model == "Swerling1"].iloc[0]
    fast = cadence_probe[cadence_probe.model == "Swerling2"].iloc[0]
    if abs(slow.realized_maximum_rcs_factor - slow.realized_minimum_rcs_factor) > 1e-12:
        raise AssertionError("Swerling I changed inside a dwell")
    if not fast.realized_maximum_rcs_factor > fast.realized_minimum_rcs_factor:
        raise AssertionError("Swerling II did not change pulse to pulse")

    validate_swerling_statistics()
    print(
        f"SELF-TEST PASSED: tracker plus Swerling distributions/cadence; "
        f"{len(recovered)} accepted synthetic track(s).",
        flush=True,
    )


def process_hour(hour_key):
    hour_out = HOURLY_ROOT / hour_key
    hour_out.mkdir(parents=True, exist_ok=True)
    complete_marker = hour_out / "COMPLETE.json"
    if complete_marker.exists():
        payload = json.loads(complete_marker.read_text(encoding="utf-8"))
        if payload.get("config_hash") == CONFIG_HASH and payload.get("status") == "complete":
            log("Already complete; skipping.", hour_key, "resume")
            return {"hour_key": hour_key, "status": "skipped_complete"}

    started = time.monotonic()
    stage_dir = None
    try:
        stage_dir, files, layout = stage_hour(hour_key)
        data = load_hour(hour_key, files)
        scores = calculate_score_fields(hour_key, data)

        if not (hour_out / "REAL_COMPLETE.json").exists():
            run_real_sweep(hour_key, data, scores, hour_out)
        else:
            log("Real sweep checkpoint exists; skipping.", hour_key, "resume")

        if not (hour_out / "NULL_COMPLETE.json").exists():
            run_null_controls(hour_key, data, scores, hour_out)
        else:
            log("Null checkpoint exists; skipping.", hour_key, "resume")

        payload = {
            "hour_key": hour_key,
            "status": "complete",
            "config_hash": CONFIG_HASH,
            "layout": layout,
            "source_hdf5_files": data["hdf5_files"],
            "integrations": int(len(data["time_unix"])),
            "range_cells": int(len(data["ranges_slant_km"])),
            "observation_hours": float(data["observation_hours"]),
            "completed_at_utc": utc_now(),
            "elapsed_seconds": time.monotonic() - started,
        }
        atomic_json(payload, complete_marker)
        log(
            f"HOUR COMPLETE in {payload['elapsed_seconds'] / 60:.1f} min.",
            hour_key,
            "complete",
        )
        del data, scores
        gc.collect()
        safe_remove_local(stage_dir)
        return payload

    except Exception as error:
        failure = {
            "hour_key": hour_key,
            "status": "failed",
            "config_hash": CONFIG_HASH,
            "failed_at_utc": utc_now(),
            "error": repr(error),
            "traceback": traceback.format_exc(),
        }
        atomic_json(failure, hour_out / "FAILURE.json")
        log(f"FAILED: {error}\n{failure['traceback']}", hour_key, "failure")
                                                                              
        return failure


run_self_test()


In [ ]:
if RESUME_REAL_ONLY:
    swerling_marker = SWERLING_ROOT / "SWERLING_COMPLETE.json"
    swerling_path = SWERLING_ROOT / "swerling_trials_all.csv.gz"
    if not swerling_marker.exists() or not swerling_path.exists():
        raise RuntimeError(
            "Fast resume requires the completed raw-Swerling existing outputs."
        )
    swerling_payload = json.loads(
        swerling_marker.read_text(encoding="utf-8")
    )
    if swerling_payload.get("config_hash") != CONFIG_HASH:
        raise RuntimeError(
            "Saved raw-Swerling results use a different configuration hash."
        )
    swerling_all = pd.read_csv(swerling_path)
    print(
        "FAST RESUME: loaded completed raw-Swerling table directly from saved outputs; "
        "no Hardtarget work was rerun.",
        flush=True,
    )
else:
    swerling_all = run_swerling_raw_calibration()
print(
    "RAW SWERLING RESULT ROWS AVAILABLE:",
    swerling_all.trial_id.nunique(),
    "independent target trials",
)
display(
    swerling_all.groupby(
        ["model", "requested_output_snr_db", "duration_sec"],
        as_index=False,
    ).recovered.mean().head(20)
)


In [ ]:
selected_hours = inventory["hour_key"].tolist()
if HOURS_TO_RUN is not None:
    requested = set(HOURS_TO_RUN)
    unknown = sorted(requested - set(selected_hours))
    if unknown:
        raise ValueError(f"Unknown HOURS_TO_RUN values: {unknown}")
    selected_hours = [hour for hour in selected_hours if hour in requested]

print("Hours selected:", selected_hours, flush=True)
print(
    "Safe to rerun: finished hour phases are checkpointed and skipped.",
    flush=True,
)

campaign_results = []
campaign_started = time.monotonic()
for hour_number, hour_key in enumerate(selected_hours, start=1):
    log(
        f"STARTING HOUR {hour_number}/{len(selected_hours)}; "
        f"campaign elapsed={(time.monotonic() - campaign_started) / 3600:.2f} h",
        hour_key,
        "campaign",
    )
    result = process_hour(hour_key)
    campaign_results.append(result)

campaign_status = pd.DataFrame(campaign_results)
display(campaign_status)
atomic_csv(campaign_status, OUTPUT_ROOT / "campaign_processing_status.csv")

failures = campaign_status[campaign_status.status == "failed"]
print("=" * 78)
print("PRODUCTION LOOP FINISHED")
print("=" * 78)
print("Elapsed hours:", (time.monotonic() - campaign_started) / 3600)
print("Status counts:")
display(campaign_status.status.value_counts())
if len(failures):
    print(
        "Some hours failed, but later hours were still attempted. "
        "Inspect hourly/<hour>/FAILURE.json and rerun this cell.",
        flush=True,
    )


In [ ]:
def read_completed_results():
    swerling_marker = SWERLING_ROOT / "SWERLING_COMPLETE.json"
    swerling_path = SWERLING_ROOT / "swerling_trials_all.csv.gz"
    if not swerling_marker.exists() or not swerling_path.exists():
        raise RuntimeError(
            "Raw Swerling calibration is incomplete. Run the raw Swerling cell first."
        )
    swerling_payload = json.loads(
        swerling_marker.read_text(encoding="utf-8")
    )
    if swerling_payload.get("config_hash") != CONFIG_HASH:
        raise RuntimeError("Raw Swerling results use a different configuration")

    real_frames = []
    null_frames = []
    completed_hours = []
    for hour_key in inventory.hour_key:
        hour_out = HOURLY_ROOT / hour_key
        marker = hour_out / "COMPLETE.json"
        if not marker.exists():
            continue
        payload = json.loads(marker.read_text(encoding="utf-8"))
        if payload.get("config_hash") != CONFIG_HASH:
            continue
        real_frames.append(pd.read_csv(hour_out / "real_summary.csv"))
        null_frames.append(pd.read_csv(hour_out / "null_trials.csv.gz"))
        completed_hours.append(hour_key)
    if not completed_hours:
        raise RuntimeError("No completed real-data benchmark hours are available")
    return (
        completed_hours,
        pd.concat(real_frames, ignore_index=True),
        pd.concat(null_frames, ignore_index=True),
        pd.read_csv(swerling_path),
    )


def poisson_rate_upper_95(count, exposure):
    if exposure <= 0:
        return np.nan
    return 0.5 * chi2.ppf(0.95, 2.0 * (count + 1.0)) / exposure


def target_performance_metrics(targets):
    fluctuating = targets[targets.model != "Swerling0"]
    focus = fluctuating[
        (fluctuating.duration_sec <= 2.0)
        & (fluctuating.requested_output_snr_db <= 14.0)
    ]
    by_model = fluctuating.groupby("model").recovered.mean()
    return {
        "pd_swerling_all": float(fluctuating.recovered.mean()),
        "pd_focus_short_weak": float(focus.recovered.mean()),
        "pd_worst_swerling_model": float(by_model.min()),
        "pd_nonfluctuating": float(
            targets.loc[targets.model == "Swerling0", "recovered"].mean()
        ),
    }


def ranking_table(targets, nulls):
    group = ["config_id", "method", "threshold", "units", "track_rule"]
    fluctuating = targets[targets.model != "Swerling0"]
    overall = (
        fluctuating.groupby(group, as_index=False)
        .agg(
            target_trials=("trial_id", "nunique"),
            pd_swerling_all=("recovered", "mean"),
        )
    )
    focus = (
        fluctuating[
            (fluctuating.duration_sec <= 2.0)
            & (fluctuating.requested_output_snr_db <= 14.0)
        ]
        .groupby(group, as_index=False)
        .agg(pd_focus_short_weak=("recovered", "mean"))
    )
    short = (
        fluctuating[fluctuating.duration_sec <= 2.0]
        .groupby(group, as_index=False)
        .agg(pd_short=("recovered", "mean"))
    )
    weak = (
        fluctuating[fluctuating.requested_output_snr_db <= 14.0]
        .groupby(group, as_index=False)
        .agg(pd_weak=("recovered", "mean"))
    )
    steady = (
        targets[targets.model == "Swerling0"]
        .groupby(group, as_index=False)
        .agg(pd_nonfluctuating=("recovered", "mean"))
    )
    by_model = (
        fluctuating.groupby(group + ["model"], as_index=False)
        .recovered.mean()
        .pivot(index=group, columns="model", values="recovered")
        .reset_index()
        .rename(
            columns={
                "Swerling1": "pd_swerling1",
                "Swerling2": "pd_swerling2",
                "Swerling3": "pd_swerling3",
                "Swerling4": "pd_swerling4",
            }
        )
    )
    model_columns = [
        "pd_swerling1", "pd_swerling2", "pd_swerling3", "pd_swerling4"
    ]
    by_model["pd_worst_swerling_model"] = by_model[model_columns].min(axis=1)
    null_summary = (
        nulls.groupby(group, as_index=False)
        .agg(
            false_tracks=("false_tracks", "sum"),
            null_exposure_hours=("null_exposure_hours", "sum"),
        )
    )
    table = (
        overall
        .merge(focus, on=group, how="left")
        .merge(short, on=group, how="left")
        .merge(weak, on=group, how="left")
        .merge(steady, on=group, how="left")
        .merge(by_model, on=group, how="left")
        .merge(null_summary, on=group, how="left")
    )
    table["false_tracks_per_hour"] = (
        table.false_tracks / table.null_exposure_hours
    )
    table["false_tracks_per_hour_upper95"] = [
        poisson_rate_upper_95(count, exposure)
        for count, exposure in zip(
            table.false_tracks, table.null_exposure_hours
        )
    ]
    table["false_tracks_per_24h_upper95"] = (
        24.0 * table.false_tracks_per_hour_upper95
    )
    table["meets_false_track_budget"] = (
        table.false_tracks_per_24h_upper95 <= FALSE_TRACK_BUDGET_PER_24H
    )
    if table.meets_false_track_budget.any():
        table["eligible_for_selection"] = table.meets_false_track_budget
        table = table.sort_values(
            [
                "eligible_for_selection",
                "pd_focus_short_weak",
                "pd_worst_swerling_model",
                "pd_swerling_all",
                "false_tracks_per_24h_upper95",
            ],
            ascending=[False, False, False, False, True],
        )
    else:
        table["eligible_for_selection"] = True
        table = table.sort_values(
            [
                "false_tracks_per_24h_upper95",
                "pd_focus_short_weak",
                "pd_worst_swerling_model",
            ],
            ascending=[True, False, False],
        )
    return table.reset_index(drop=True)


completed_hours, real_all, null_all, swerling_all = read_completed_results()
log(
    f"Aggregating {len(completed_hours)}/24 real hours and "
    f"{swerling_all.trial_id.nunique()} raw target trials.",
    step="aggregate",
)

ranking = ranking_table(swerling_all, null_all)
atomic_csv(ranking, OUTPUT_ROOT / "detector_operating_point_ranking.csv")

if len(completed_hours) != 24:
    raise RuntimeError(
        f"Only {len(completed_hours)}/24 real hours are complete. A partial ranking "
        "was saved, but held-out selection is blocked. Rerun the real-data cell; "
        "completed hours will be skipped."
    )

                                                                 
                                       
hour_to_fold = {
    hour_key: index // 4
    for index, hour_key in enumerate(inventory.hour_key.tolist())
}
null_all["fold"] = null_all.hour_key.map(hour_to_fold)

fold_rows = []
for held_out_fold in range(6):
    train_targets = swerling_all[swerling_all.fold != held_out_fold]
    test_targets = swerling_all[swerling_all.fold == held_out_fold]
    train_null = null_all[null_all.fold != held_out_fold]
    test_null = null_all[null_all.fold == held_out_fold]
    train_ranking = ranking_table(train_targets, train_null)
    chosen = train_ranking.iloc[0]
    held_targets = test_targets[
        (test_targets.config_id == chosen.config_id)
        & (test_targets.track_rule == chosen.track_rule)
    ]
    held_null = test_null[
        (test_null.config_id == chosen.config_id)
        & (test_null.track_rule == chosen.track_rule)
    ]
    metrics = target_performance_metrics(held_targets)
    false_count = int(held_null.false_tracks.sum())
    exposure = float(held_null.null_exposure_hours.sum())
    fold_rows.append(
        {
            "held_out_fold": held_out_fold,
            "held_out_hours": "+".join(
                hour for hour, fold in hour_to_fold.items()
                if fold == held_out_fold
            ),
            "chosen_config_id": chosen.config_id,
            "chosen_method": chosen.method,
            "chosen_threshold": chosen.threshold,
            "chosen_track_rule": chosen.track_rule,
            "test_target_trials": int(held_targets.trial_id.nunique()),
            **metrics,
            "test_false_tracks": false_count,
            "test_null_exposure_hours": exposure,
            "test_false_tracks_per_hour": false_count / exposure,
            "test_false_tracks_per_24h_upper95": (
                24.0 * poisson_rate_upper_95(false_count, exposure)
            ),
        }
    )

fold_results = pd.DataFrame(fold_rows)
atomic_csv(fold_results, OUTPUT_ROOT / "six_fold_held_out_evaluation.csv")

winner = ranking.iloc[0].to_dict()
atomic_json(
    {
        "selected_at_utc": utc_now(),
        "config_hash": CONFIG_HASH,
        "completed_hours": completed_hours,
        "raw_swerling_trials": int(swerling_all.trial_id.nunique()),
        "selection": winner,
        "selection_rule": (
            "Among operating points whose 95% upper false-track bound is "
            "<= 1 per 24 hours, maximize raw pre-GMF recovery for short "
            "(<=2 s), weak/moderate (<=14 dB nonfluctuating-reference output) "
            "Swerling I-IV targets; break ties by worst Swerling-model Pd, "
            "then all-grid Swerling Pd, then the smaller false-track bound."
        ),
        "scope": (
            "Detector and tracker operating point calibrated with complex raw "
            "simulation and real-data nulls. Diameter/RCS sensitivity, catalog "
            "association, and raw-over-real-background spot checks are separate."
        ),
    },
    OUTPUT_ROOT / "SELECTED_DETECTOR.json",
)

print("=" * 78)
print("RAW-SWERLING / REAL-NULL DETECTOR RANKING")
print("=" * 78)
display(ranking.head(20))
print("\nSix-fold held-out evaluation:")
display(fold_results)
print("\nSelected operating point:")
display(pd.DataFrame([winner]))

fig, axis = plt.subplots(1, 1, figsize=(8, 6))
for method, subset in ranking.groupby("method"):
    axis.scatter(
        subset.false_tracks_per_24h_upper95,
        subset.pd_focus_short_weak,
        label=method,
        alpha=0.75,
    )
axis.axvline(FALSE_TRACK_BUDGET_PER_24H, color="black", linestyle="--")
axis.set_xlabel("95% upper false tracks per 24 h")
axis.set_ylabel("Raw pre-GMF Pd: short, weak Swerling I-IV")
axis.set_title("Detector operating-point tradeoff")
axis.legend()
axis.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    OUTPUT_ROOT / "detector_selection_tradeoff.png",
    dpi=240,
    bbox_inches="tight",
)
plt.show()

selected_targets = swerling_all[
    (swerling_all.config_id == winner["config_id"])
    & (swerling_all.track_rule == winner["track_rule"])
    & (swerling_all.model != "Swerling0")
]
fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)
last_image = None
for axis, model in zip(axes.ravel(), ["Swerling1", "Swerling2", "Swerling3", "Swerling4"]):
    heatmap = selected_targets[selected_targets.model == model].pivot_table(
        index="duration_sec",
        columns="requested_output_snr_db",
        values="recovered",
        aggfunc="mean",
    )
    last_image = axis.imshow(
        heatmap.values,
        origin="lower",
        aspect="auto",
        vmin=0,
        vmax=1,
        cmap="viridis",
    )
    axis.set_xticks(range(len(heatmap.columns)), heatmap.columns)
    axis.set_yticks(range(len(heatmap.index)), heatmap.index)
    axis.set_title(
        f"{model}: {swerling_update_rate(model)}"
    )
    axis.set_xlabel("Nonfluctuating-reference output SNR (dB)")
    axis.set_ylabel("Illumination-track duration (s)")
fig.colorbar(
    last_image,
    ax=axes.ravel().tolist(),
    label="Track recovery probability",
    shrink=0.85,
)
fig.suptitle(
    f"Selected {winner['method']} operating point: raw pre-GMF Swerling recovery",
    y=1.01,
)
plt.savefig(
    OUTPUT_ROOT / "swerling_model_recovery_panels.png",
    dpi=240,
    bbox_inches="tight",
)
plt.show()


## Outputs and interpretation
